# Exploration de Tox21

Charge les partitions officielles MoleculeNet avec DeepChem. La première exécution peut télécharger le jeu Tox21. Les partitions train/validation/test sont celles du loader (scaffold), pas un split aléatoire ajouté par ce notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.tox21 import load_tox21_splits, write_tox21_audit

splits = load_tox21_splits()
report = write_tox21_audit(splits)
[(name, len(dataset)) for name, dataset in zip(("train", "validation", "test"), splits.datasets)]

In [ ]:
print("Tâches :", report["tasks"])
print("Features :", report["feature_object_type"])
print("IDs are SMILES :", report["ids"]["contains_smiles"])
print("Transforms :", report["transformers"])
print("Split :", report["splitter"], "| seed:", report["split_seed"])

summary = pd.DataFrame({
    split_name: {
        task: f"{counts['positive']} / {counts['negative']} / {counts['missing']}"
        for task, counts in split["tasks"].items()
    }
    for split_name, split in report["splits"].items()
})
summary.index.name = "task (positifs / négatifs / masqués)"
summary

Dans DeepChem, `y` contient les valeurs d'essai et `w == 0` marque une cible manquante/non observée. Pour toute perte ou métrique, utiliser `w` comme masque : ne pas interpréter le zéro de remplissage de `y` comme un résultat négatif. Ne jamais ajuster un modèle ni sélectionner des hyperparamètres avec le split test.

In [ ]:
train = splits.train
print("type X[0] :", type(train.X[0]))
print("shapes : X", train.X.shape, "y", train.y.shape, "w", train.w.shape, "ids", train.ids.shape)
print("exemples d'IDs/SMILES :", train.ids[:3])